# 1장 1강: 문제 유형·데이터 분리·평가지표 리마인드

## 실습 목표

- Ames Housing의 `SalePrice`를 예측하는 문제가 어떤 머신러닝 문제 유형인지 구분합니다.
- 입력 데이터 `X`와 정답 `y`를 구분합니다.
- 데이터를 `train / valid / test`로 분리하고 각 데이터의 역할을 확인합니다.
- 회귀 평가지표 `MSE`, `RMSE`, `MAE`, `R²`의 의미를 확인합니다.
- 교안에서 소개한 **평균 예측 baseline**을 기준으로 기본 성능을 해석합니다.

> **실습 범위 안내**  
> 이번 실습은 1장 1강 교안의 리마인드 범위만 사용합니다.  
> 별도의 회귀 알고리즘, 교차검증, 하이퍼파라미터 튜닝, 새로운 평가지표는 사용하지 않습니다.

## 실습 환경 / 데이터

- Python
- pandas, numpy, scikit-learn
- 데이터: `Ames_Housing.csv`
- 예측 대상: `SalePrice` (주택 판매 가격)
- 이번 실습에서 확인할 입력 컬럼: `GrLivArea`, `OverallQual`, `YearBuilt`


## 실습 준비

먼저 데이터를 불러온 뒤 이번 실습에서 사용할 컬럼을 확인합니다.

- `GrLivArea`: 지상 생활 면적
- `OverallQual`: 주택의 전반적인 품질
- `YearBuilt`: 건축 연도
- `SalePrice`: 주택 판매 가격

이번 강의의 핵심은 **문제 유형, 데이터 분리, 평가지표**이므로 복잡한 전처리는 진행하지 않습니다.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("Ames_Housing.csv")

use_cols = ["GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]
data = df[use_cols].copy()

print("데이터 크기:", data.shape)
display(data.head())


데이터 크기: (1460, 4)


,GrLivArea,OverallQual,YearBuilt,SalePrice
0,1710,7,2003,208500
1,1262,6,1976,181500
2,1786,7,2001,223500
3,1717,7,1915,140000
4,2198,8,2000,250000


In [2]:
print("[자료형]")
display(data.dtypes.to_frame("dtype"))

print("[결측치 수]")
display(data.isna().sum().to_frame("missing_count"))

print("[기초 통계량]")
display(data.describe())


[자료형]


,dtype
GrLivArea,int64
OverallQual,int64
YearBuilt,int64
SalePrice,int64


[결측치 수]


,missing_count
GrLivArea,0
OverallQual,0
YearBuilt,0
SalePrice,0


[기초 통계량]


,GrLivArea,OverallQual,YearBuilt,SalePrice
count,1460.000000,1460.000000,1460.000000,1460.000000
mean,1515.463699,6.099315,1971.267808,180921.195890
std,525.480383,1.382997,30.202904,79442.502883
min,334.000000,1.000000,1872.000000,34900.000000
25%,1129.500000,5.000000,1954.000000,129975.000000
50%,1464.000000,6.000000,1973.000000,163000.000000
75%,1776.750000,7.000000,2000.000000,214000.000000
max,5642.000000,10.000000,2010.000000,755000.000000


In [3]:
# 지도학습 : X와 정답 Y를 함께 사용해 둘 사이의 관계를 학습하는 방법
# -> 이커머스에서 고객 이탈여부(0, 1, 분류) / 이커머스에서 가을철 옷 매출 예측(회귀)
# -> 로지스틱 회귀, 랜덤포레스트, Regression

# 비지도학습 : 미리 정해진 정답 Y없이, 데이터의 구조나 패턴을 찾는 방법
# -> 고객군을 연령, 성별, 구매금액 등 그때그때 유사한 패턴이나 구조를 가진 사람끼리 묶는 것
# -> 군집화(K-MEANS, DBSCAN), 차원축소(PCA)

# Train / Validation / Test
# -> Train : 모델이 예측 관계나 패턴을 학습하는데 사용하는 데이터 
# -> Validation : 개발 중 모델과 설정을 비교하고 선택하는데 사용하는 데이터
# -> Test : 모든 선택을 마친 모델의 성능을 최종적으로 평가하기 위해 남겨둔 데이터

# 파라미터 / 하이퍼 파라미터
# 파라미터 : 학습으로 정해지는 모델의 내부 값
# 하이퍼 파라미터 : 학습 방식을 제어하는 설정 값
# 하이퍼 파라미터 튜닝 : 검증 성능 등을 기준으로 적절한 하이퍼 파라미터를 찾는 과정
# 모델 고도화 : 데이터 전처리 / 하이퍼 파라미터 / 외부 데이터 Join

# 예측 오차 : 실제값과 예측값 사이의 차이
# 1 -> 2 -> 4 -> 8 -> 16 -> ? # 2배씩 증가하는 패턴을 보였는데 실제 값이 아닌 경우

# 1 -> 2 -> 4 -> 8 -> 16 -> 199
# 199 - 32 = 167

# 실제 구매액 10만원
# 1번 예측 12만원 = -2만원 오차
# 2번 예측 8만원 = 2만원 오차
# -2 + 2 = 0

# 평균 절대 오차(MAE) : 각 예측 오차의 절댓값을 평균한 값
# -> 실제 구매액이 10, 20, 30, 40 만원이고 예측값이 12, 18, 33, 39 만원의 차이면
# 2, 2, 3, 1만큼 차이가 나기 때문에 MAE는 2다

# 평균 제곱 오차(MSE) : 각 예측 오차를 제곱한 뒤 평균한 값
# 구매액의 예측 오차 크기가 2, 2, 3, 1만원이면 제곱하면 4, 4, 9, 1로
# MSE는 4.5이다
# 1 -> 2 -> 4 -> 8 -> 16 -> 32
# 1 -> 2 -> 5 -> 8 -> 16 -> 35
# 3번째의 오차는 1, 6번째의 오차는 3
# 1, 9

# 평균 제곱근 오차(RMSE) : MSE에 제곱근을 적용한 값
# 네 고객의 구매액을 A모델은 각 2만원씩 틀리고, B모델은 세 명을 맞히고 8만원 정도 틀렸다면
# MAE는 둘다 2만원이지만 RMSE는 A가 2만원 B가 4만원이다.
# B가 조금 더 큰 실수가 있었다는 것을 알 수 있다.

# 결정계수 R^2 : 평가 데이터의 실제값 평균으로 예측하는 기준과 모델의 제곱 오차를 비교한 지표(설명력)
# -> 실제 구매액 10, 20, 30, 40 만원을 모두 평균인 25만원으로 예측하면 제곱 오차 합은 500
# -> 모델이 12, 18, 33, 39만원으로 예측하면 합은 18이기 떄문에 R^2은 1 - 18 / 500 = 0.964
# 오차 합이 96.4%

# 혼동행렬 : 실제 클래스와 예측 클래스를 교차해 맞힌 경우와 틀린 경우를 세는 표
# 이진 분류에서 TP, TN, FP, FN의 네 칸으로 결과를 구분한다.

# TP : 실제 1(P)을 1(P)로 맞힌 경우
# TN : 실제 0(N)을 0(N)으로 맞힌 경우
# -> 구매 예측에서 실제 구마핸 고객을, 구매한 고객으로 정확히 예측한 15명을 TP로
# -> 실제 구매하지 않은 고객 70명을 TN으로, 둘의 합인 85명은 올바르게 에측한 고객이다.

# FP : 실제 0(N)인데 1(P)로 잘못 예측한 경우
# 불량을 양성으로 정했을 때 정상 제품 10개를 불량이라고 판정하면 FP는 10개
# 정상 제품을 재검사 대상으로 잘못 판단

# FN : 실제 1(P)인데 0(N)으로 잘못 예측한 경우
# 불량 제품 20개 중 10개를 정상이라고 잘못 판정하는 경우
# 재검사 대상으로 판단해야하는데 잘못 판단

# Accuracy(정확도) : 전체 평가 사례 중 올바르게 예측한 비율
# -> TP와 TN을 합쳐 전체 수로 나눈다.
# -> 구매 예측 100건에서 구매를 맞힌 15건과 미구매를 맞힌 70건을 합치면 약 85%의 정확도가 나온다.
# -> 100개의 기업 중 94개 정도가 폐업을 했다. -> 100개 전체가 다 폐업됐다고 예측을 해도 정확도는 94%가 나온다는 허점이 존재
# 단독으로는 사용을 권장하지 않고 다른 성능 평가지표들과 함께 사용한다.

# 클래스 불균형 : 정답 범주별 데이터 수가 크게 다른 상태
# -> 고객 100명중 구매자 5명, 미구매자 95명일 때 -> 클래스 불균형 상태다.

# Precision(정밀도) : 양성(1, P)라고 예측한 것중에 실제 (1,P)인 비율
# -> 분모는 양성이라고 예측한 전체, 분자는 실제 양성인 것
# -> 구매할 것이라고 예측한 고객 25명 중 실제 구매자가 15명이면 15/25 -> 60%의 정밀도를 보인다.
# 정밀도는 양성이라고 예측한 대상 중 실제 양성의 비율, 즉 정답률을 확인하기 위함

# Recall(재현율) : 실제 (1,P)인 것 중 모델이 양성(1,P)으로 찾아낸 비율
# -> 분모가 실제 양성인 것 전체, 분자가 예측한 실제 양성
# -> 실제 구매자 20명 중 15명을 구매자로 찾아냈다면 재현율은 75%이다.
# 재현율은 실제 양성을 얼마나 놓치지 않았는지, 즉 오답률을 확인하기 위함

# F1-Score : 정밀도와 재현율의 조화평균
# 위에서 정밀도가 60%이고 재현율이 75%라면 F1-Score는 67% 정도이다.

# ROC 곡선 : 임계값 변화에 따른 양성 탐지율과 음성 오판율을 보기 위함

# ROC - AUC : 곡선 아래 면적으로 양성과 음성을 점수로 구분하는 능력을 요약
# -> 실제 구매자 두명의 점수가 0.9, 0.6이고 미구매자 두 명의 점수가 0.7과 0.2라면 구매자, 미구매자 쌍 4쌍을 만들 수 있는데
# -> 세 쌍에서 구매자의 점수가 높아서 0.75의 AUC를 가진다.


---

# 필수 1. 문제 유형과 train / valid / test 분리

## 배경

Ames Housing 데이터에서 주택의 특성을 이용해 `SalePrice`를 예측하려고 합니다.

교안에서 머신러닝 문제는 회귀·분류·군집화·차원축소로 구분했고, 지도학습에서는 입력 `X`와 정답 `y`를 사용합니다. 또한 모델의 일반화 성능을 확인하기 위해 데이터를 `train / valid / test`로 나눕니다.

## 문제

### 요구사항

1. `SalePrice` 예측이 회귀·분류·군집화·차원축소 중 어떤 문제인지 작성하세요.
2. 아래 기준으로 `X`와 `y`를 만드세요.
   - X: `GrLivArea`, `OverallQual`, `YearBuilt`
   - y: `SalePrice`
3. 교안의 예시와 같이 데이터를 약 **6 : 2 : 2** 비율의 `train / valid / test`로 나누세요.
   - 첫 번째 분리: `test_size=0.2`, `random_state=42`
   - 두 번째 분리: 남은 데이터에서 `test_size=0.25`, `random_state=42`
4. `X_train`, `X_valid`, `X_test`의 행 개수를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** `SalePrice` 예측은 왜 해당 문제 유형에 속하나요?
- **Q2.** train 데이터는 어떤 목적으로 사용하나요?
- **Q3.** valid 데이터는 어떤 목적으로 사용하나요?
- **Q4.** test 데이터는 어떤 목적으로 사용하나요?

### 확인 포인트

- 정답 `y`가 존재하는 지도학습 문제인지 확인했는가?
- `SalePrice`가 연속적인 숫자 값이라는 점을 근거로 문제 유형을 판단했는가?
- train / valid / test의 역할을 구분했는가?


In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd

# TODO 1. X와 y를 만드세요.
X = df[['GrLivArea', 'OverallQual', 'YearBuilt']]
y = df['SalePrice']

# TODO 2. train+valid / test로 먼저 분리하세요.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,              # 전체 데이터를 넣음
    test_size=0.2,     # 전체의 20%를 test로
    random_state=42    # 결과 고정
)

# TODO 3. 남은 데이터를 train / valid로 분리하세요.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,    # 전체가 아니라 "남은 것"을 넣음
    test_size=0.25,    # 남은 것의 25% = 전체의 20%를 valid로
    random_state=42
)

# TODO 4. 각 데이터의 행 개수를 출력하세요.
print("train:", len(X_train))
print("valid:", len(X_valid))
print("test :", len(X_test))

train: 876
valid: 292
test : 292


### 필수 1 답변 작성

- **Q1. 문제 유형과 이유: SalePrice라는 정답이 주어진 지도학습(회귀). 가격이라는 연속적인 값을 예측해야하므로** 
- **Q2. train의 역할:모델이 X와 y 사이의 관계(패턴)를 학습하는 데 사용한다.** 
- **Q3. valid의 역할:학습에 쓰지 않은 데이터로 성능을 중간 점검하고, 그 결과로 여러 모델을 비교하거나 설정을 조정하는 데 사용한다. 과적합 여부도 확인** 
- **Q4. test의 역할:모든 선택이 끝난 뒤 마지막에 한 번만 사용해, 처음 보는 데이터에 대한 최종 성능(일반화 성능)을 확인한다.** 

---

# 필수 2. 평균 예측 baseline과 회귀 평가지표

## 배경

교안에서는 회귀 문제를 `MSE`, `RMSE`, `MAE`, `R²`로 평가할 수 있다고 배웠습니다. 또한 R²를 설명하면서 **아무 특성도 사용하지 않고 y의 평균값만 예측하는 기준 모델(baseline)**을 소개했습니다.

이번 문제에서는 새로운 모델 알고리즘을 사용하지 않고, **train의 SalePrice 평균을 모든 valid 데이터의 예측값으로 사용하는 baseline**을 만들어 네 가지 회귀 평가지표를 확인합니다.

## 문제

### 요구사항

1. `y_train`의 평균을 `baseline_value`에 저장하세요.
2. `baseline_value`를 valid 데이터 개수만큼 반복하여 `y_valid_pred`를 만드세요.
3. `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 네 가지 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** MSE는 무엇을 의미하나요?
- **Q2.** RMSE는 무엇을 의미하며 MSE와 어떤 차이가 있나요?
- **Q3.** MAE는 무엇을 의미하나요?
- **Q4.** R²가 0에 매우 가깝다면 이 baseline을 어떻게 해석할 수 있나요?
- **Q5.** RMSE가 MAE보다 크게 나오는 이유를 교안의 설명을 이용해 작성하세요.

### 확인 포인트

- baseline 값은 반드시 `y_train`의 평균으로 만들었는가?
- MSE, RMSE, MAE, R²만 사용했는가?
- RMSE가 큰 오차에 더 큰 영향을 받는다는 점을 설명했는가?


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# TODO 1. y_train 평균으로 baseline 값을 만드세요.
baseline_value = y_train.mean()

# TODO 2. valid 데이터 전체에 같은 baseline 값을 예측하도록 만드세요.
y_valid_pred = np.full(len(y_valid), baseline_value)

# TODO 3. MSE, RMSE, MAE, R²를 계산하세요.
mse = mean_squared_error(y_valid, y_valid_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_valid, y_valid_pred)
r2 = r2_score(y_valid, y_valid_pred)

# TODO 4. 결과를 출력하세요.
print("baseline:", baseline_value)
print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)

baseline: 181233.96803652967
MSE : 5989110885.717651
RMSE: 77389.34607371774
MAE : 57793.59893976355
R²  : -0.00011512063522900462


### 
**해설**
- Baseline : 181,233달러 / 모든 검증 주택에 동일하게 제시한 예측 가격
- MSE : 5,989,110,885달러 / 제곱한 오차의 평균(낮을수록 좋음)
- RMSE : 77,389달러 / 큰 오차를 더 강하게 반영한 오차 규모(낮을수록 좋음) -> 주택당 77,389달러 정도 틀릴 수 있다.
- MAE : 57,793 달러 / 절대 오차의 평균(낮을수록 좋음) -> 전체적으로 57,793달러 정도 틀릴 수 있다.
- R^2 : -0.00015 / 평가 데이터 평균을 예측하는 기준에 비해 얼마나 개선되었는가? (높을수록 좋음), 설명력

###
- 이 Baseline은 면적이나 품질, 건축연도를 사용하지 않아 주택별 가격 차이를 충분히 설명하지 못함, 완성된 가격 예측 방법의 우수성을 보여주는 결과가 아니라 이후 방법을 평가할 기준 성능을 의미

- 학습 데이터의 평균 가격인 181,233 달러를 모든 검증 주택 예측에 사용했고, 검증 MAE는 약 57,793달러, RMSE는 약 77,389달러, R2는 -0.00015으로, 데이터 평균을 예측하는 기준과 거의 같은 수준

### 필수 2 답변 작성

- **Q1. MSE:실제값과 예측값의 오차를 제곱한 뒤 평균한 값, 큰 오차가 있으면 숫자가 커짐** 
- **Q2. RMSE:MSE에 제곱근을 취한 값으로 원래 단위로 해석하여 오차를 직관적으로 볼 수 있음** 
- **Q3. MAE:실제값과 예측값 차이의 절댓값을 평균한 값, 평균적으로 얼마나 벗어났는지 보여줌** 
- **Q4. R² 해석:특성을 사용하지 않고 평균값만 예측하는 기준 모델 수준과 다르지 않음, 0에 가깝거나 음수라는 것은 이 모델이 SalePrice의 차이를 거의 설명하지 못한다는 의미이다.** 
- **Q5. RMSE와 MAE 차이 해석:RMSE는 오차를 제곱하여 크게 틀린 경우에 더 민감하게 반응하고 큰 오차가 존재하면 MAE보다 더 값이 크게 나올 수 있음** 


---

# 과제 1. 테스트 데이터에서 baseline 최종 확인

## 배경

교안에서는 test 데이터를 **최종 성능을 마지막에 한 번 확인하는 데이터**라고 설명했습니다.

필수 2에서 사용한 평균 예측 baseline을 그대로 유지한 상태에서 test 데이터의 성능을 확인해보세요. 새로운 모델이나 새로운 평가 방법은 사용하지 않습니다.

## 문제

### 요구사항

1. 필수 2에서 만든 `baseline_value`를 그대로 사용하세요.
2. test 데이터 개수만큼 동일한 값을 예측하여 `y_test_pred`를 만드세요.
3. test 데이터에 대해 `MSE`, `RMSE`, `MAE`, `R²`를 계산하세요.
4. 결과를 출력하세요.
5. 아래 질문에 답하세요.

### Q&A

- **Q1.** 이번 과제에서 test 데이터를 사용하는 목적은 무엇인가요?
- **Q2.** test의 R²가 0에 가깝거나 음수라면 평균 예측 baseline의 성능을 어떻게 해석할 수 있나요?
- **Q3.** RMSE와 MAE 중 큰 오차에 더 큰 벌점을 주는 지표는 무엇이며, 왜 그런가요?

### 확인 포인트

- test 데이터에서 새로운 값을 학습하거나 기준을 다시 정하지 않았는가?
- `y_train`에서 구한 baseline 값을 그대로 사용했는가?
- 교안의 회귀 평가지표 범위만 사용했는가?


In [6]:
# TODO 1. test 데이터의 baseline 예측값을 만드세요.
y_test_pred = np.full(shape=len(y_test), fill_value=baseline_value)

# TODO 2. test 데이터의 MSE, RMSE, MAE, R²를 계산하세요.
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# TODO 3. 결과를 출력하세요.
print("=== Baseline (Train Mean) Test Data Evaluation ===")
print(f"MSE  : {test_mse:,.2f}")
print(f"RMSE : {test_rmse:,.2f}")
print(f"MAE  : {test_mae:,.2f}")
print(f"R²   : {test_r2:.4f}")


=== Baseline (Train Mean) Test Data Evaluation ===
MSE  : 7,676,058,192.02
RMSE : 87,613.12
MAE  : 62,516.21
R²   : -0.0007


### 과제 1 답변 작성

- **Q1. test 데이터의 목적:** 모델이 처음 보는 데이터를 가지고 최종 성능을 점검하기 위함.
- **Q2. R² 해석:** 0에 가까울수록 설명력이 낮다는 뜻으로, 음수라면 아예 데이터를 파악하지 못하고 있다고 볼 수 있다. 대충 평균때린 것 만 못하다는 뜻.
- **Q3. 큰 오차에 더 큰 벌점을 주는 지표와 이유:** RMSE -> 오차 제곱의 평균에 제곱근을 취한 것으로, 단순 절댓값보다 큰 오차에 더 민감하다.

---

# 실습 마무리

아래 세 가지를 한 문장씩 정리해보세요.

1. **어떤 분석 문제가 있었는가?**
- 가장 먼저, 분류, 회귀, 군집화 중 어는 것을 적용해야하는지 확인이 필요했고 SalesPrice를 예측해야하는 회귀 문제였다.

2. **어떤 데이터 분리와 평가지표를 적용했는가?**
- train / valid / test로 나누고 MAE, MSE, RMSE, R2를 사용했다 

3. **평균 예측 baseline의 결과를 무엇을 근거로 해석했는가?**
- train의 평균값을 사용하는 baseline의 valid, test 평가 결과와 각 회귀 성능 평가지표의 의미를 근거로 기준 성능을 확인했다.